# Sprint 1: Datos, Calibracion y Trazabilidad

Este sprint prepara Common Voice 17.0 en espanol para dos ramas independientes:

1. `train`: adaptacion monolingue de Sprint 1.5.
2. `validation`: calibracion PTQ de los sprints de cuantizacion.

El audio de Common Voice puede variar en frecuencia de muestreo. Se convierte a mono,
24 kHz y -25 dBFS antes de construir el tensor de calibracion. El split `test` queda
reservado para el benchmark final y no se usa para entrenar ni calibrar.

## Orden de ejecucion: Sprint 1 y Sprint 1.5

La linea reproducible tiene un unico fine-tuning:

`weights/vibevoice-1.5b` -> LoRA + diffusion head -> `weights/vibevoice-1.5b-es`

| ID | Ejecuta | Artefacto o validacion |
|---|---|---|
| `S1-00` | Notebook | Entorno, semillas y rutas |
| `S1-01` | Notebook | Common Voice 17 ES fijado por revision |
| `S1-02` | Notebook | Auditoria opcional y determinista |
| `S1-03` | Notebook | Asignacion de splits nativos |
| `S1-04` | Notebook | Trazabilidad de splits |
| `S1-05` | Notebook | Preprocesamiento 24 kHz / -25 dBFS |
| `S1-06` | Notebook | Calibracion estratificada de 512 clips |
| `S1-07` | Notebook | Tensor y metadatos de calibracion |
| `S1-08` | Notebook | Verificacion del set de calibracion |
| `S1.5-01` | Notebook | Manifiestos train/dev speaker-disjoint |
| `S1.5-02` | Notebook | Validar `scripts/run_finetune_es.sh` |
| `S1.5-03` | Job persistente | Entrenar y validar el primer LoRA |
| `S1.5-04` | Notebook | Validar `scripts/merge_es_checkpoint.sh` |
| `S1.5-05` | Job persistente | Merge atomico a `vibevoice-1.5b-es` |
| `S1.5-06` | Notebook | Carga BF16 del modelo fusionado |
| `S1.5-07` | Job persistente | Gate TTS + Whisper WER/CER con voz |

No avance a cuantizacion hasta que `S1.5-07` apruebe `WER <= 0.30` y se escuchen los WAV.

## S1-00: Entorno y rutas

El notebook debe abrirse desde la raiz del repositorio o desde `notebooks/`. Los procesos
largos usan el mismo interprete del kernel mediante `sys.executable`; no hay rutas a un
entorno personal. El parche de `CONFIG_MAPPING` evita la colision conocida entre
Transformers 4.51.3 y la configuracion acustica de VibeVoice.

In [ ]:
print("[S1-00] Configuracion reproducible del entorno")
import gc, hashlib, json, os, random, sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import transformers
from datasets import load_dataset
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in [notebook_cwd, notebook_cwd.parent]
     if (candidate / "VibeVoice_repo").is_dir() and (candidate / "scripts").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz del proyecto desde {notebook_cwd}")

DATA_DIR = PROJECT_ROOT / "data"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
for directory in [DATA_DIR, WEIGHTS_DIR, OUTPUTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_ID = "fsicoli/common_voice_17_0"
DATASET_CONFIG = "es"
DATASET_REVISION = "8262c16bf297c87a9cd88c51997c4758ed7a8ba2"
BASE_MODEL_REPO = "vibevoice/VibeVoice-1.5B"
BASE_MODEL_REVISION = "d374386b2a51d8e05277a64d85b296c89ec52376"
BASE_MODEL_PATH = WEIGHTS_DIR / "vibevoice-1.5b"
if not (BASE_MODEL_PATH / "config.json").is_file():
    print(f"Descargando {BASE_MODEL_REPO}@{BASE_MODEL_REVISION}...")
    snapshot_download(
        repo_id=BASE_MODEL_REPO, revision=BASE_MODEL_REVISION,
        local_dir=BASE_MODEL_PATH,
    )
assert (BASE_MODEL_PATH / "config.json").is_file() and any(BASE_MODEL_PATH.glob("*.safetensors"))

print(f"Python       : {sys.version.split()[0]} ({sys.executable})")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
print(f"Proyecto     : {PROJECT_ROOT}")

---
## TSK-1.1: Descarga de Common Voice 17.0 ES

Se fija la revision del mirror comunitario para que los registros y su orden no cambien.
Solo se descargan los splits utilizados: `train`, `validation` y `test`; `other` e
`invalidated` no forman parte de este pipeline.

In [ ]:
print("[S1-01] Carga de Common Voice 17 espanol")
train_dataset, val_dataset, test_dataset = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    revision=DATASET_REVISION,
    split=["train", "validation", "test"],
    trust_remote_code=True,
)
cv17_dataset = train_dataset
print(f"Train      : {len(train_dataset):,} | fingerprint={train_dataset._fingerprint}")
print(f"Validation : {len(val_dataset):,} | fingerprint={val_dataset._fingerprint}")
print(f"Test       : {len(test_dataset):,} | fingerprint={test_dataset._fingerprint}")
print(f"Columnas   : {train_dataset.column_names}")

---
## Exploracion del Dataset

Inspeccionamos la estructura, metadatos demograficos y distribucion de duraciones
para validar la calidad de la muestra antes de la particion.

In [ ]:
print("[S1-02] Exploracion opcional y determinista")
audit_rng = random.Random(SEED)
sample_count = min(5000, len(train_dataset))
audit_indices = audit_rng.sample(range(len(train_dataset)), sample_count)
durations, genders, ages = [], Counter(), Counter()
for idx in tqdm(audit_indices, desc="Auditando train"):
    row = train_dataset[idx]
    audio = row["audio"]
    durations.append(len(audio["array"]) / audio["sampling_rate"])
    genders[row.get("gender") or "unknown"] += 1
    ages[row.get("age") or "unknown"] += 1

for idx in audit_indices[:3]:
    row = train_dataset[idx]
    audio = row["audio"]
    print(f"[{idx}] {row['sentence']}")
    print(f"  sr={audio['sampling_rate']} Hz | duracion={len(audio['array']) / audio['sampling_rate']:.2f}s")
print(f"Duracion media: {np.mean(durations):.2f}s | estimado train: {sum(durations)/sample_count*len(train_dataset)/3600:.1f}h")
print(f"Genero: {dict(genders)}")
print(f"Edad: {dict(ages)}")

---
## TSK-1.2: Particion de la Muestra

Common Voice 17.0 ya proporciona divisiones nativas. Las usamos directamente
para mantener trazabilidad con el estandar de la comunidad:

| Particion | Split CV17 | Proposito en el proyecto |
|-----------|-----------|---------------------------|
| Test      | `test`    | Benchmark final (Sprint 5) |
| Validacion| `validation` + muestra `other` | Calibracion offline (GPTQ/AWQ) |
| Train     | `train`   | Fine-tuning espanol (Sprint 1.5) |

Cargamos todos los splits y guardamos los indices para trazabilidad.

In [ ]:
print("[S1-03] Asignacion de splits nativos")
print(f"Train (fine-tuning)      : {len(train_dataset):,} registros")
print(f"Validation (calibracion) : {len(val_dataset):,} registros")
print(f"Test (benchmark final)   : {len(test_dataset):,} registros")

In [ ]:
print("[S1-04] Guardado de trazabilidad de splits")
partitions_meta = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "splits": {
        "train": {"count": len(train_dataset), "fingerprint": train_dataset._fingerprint, "use": "fine_tuning"},
        "validation": {"count": len(val_dataset), "fingerprint": val_dataset._fingerprint, "use": "ptq_calibration"},
        "test": {"count": len(test_dataset), "fingerprint": test_dataset._fingerprint, "use": "final_benchmark"},
    },
}
partitions_path = DATA_DIR / "dataset_partitions.json"
temporary = partitions_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(partitions_meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
temporary.replace(partitions_path)
print(f"Metadatos guardados en: {partitions_path}")

---
## TSK-1.3: Preprocesamiento acustico

La calibracion usa audio crudo mono a 24 kHz y -25 dBFS. Se rechazan arrays vacios,
no finitos, silenciosos o menores de 0.5 segundos. El entrenamiento reutiliza las mismas
reglas desde `vibevoice.finetune.data_vibevoice`.

In [ ]:
print("[S1-05] Preprocesamiento de audio a 24 kHz y -25 dBFS")
import librosa

TARGET_SR = 24000
TARGET_DB_FS = -25.0
MIN_DURATION_S = 0.5
EPS = 1e-6

def preprocess_audio(audio_array: np.ndarray, orig_sr: int) -> np.ndarray | None:
    if audio_array is None:
        return None
    audio = np.asarray(audio_array, dtype=np.float32)
    if audio.ndim == 2:
        audio = audio.mean(axis=0 if audio.shape[0] <= 2 else 1)
    audio = audio.reshape(-1)
    if audio.size == 0 or not np.isfinite(audio).all():
        return None
    if orig_sr != TARGET_SR:
        audio = librosa.resample(y=audio, orig_sr=int(orig_sr), target_sr=TARGET_SR)
    if len(audio) / TARGET_SR < MIN_DURATION_S:
        return None
    rms = float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))
    if rms < EPS:
        return None
    audio *= (10.0 ** (TARGET_DB_FS / 20.0)) / rms
    peak = float(np.max(np.abs(audio)))
    if peak > 1.0:
        audio /= peak
    return audio.astype(np.float32, copy=False)

for index in range(5):
    sample = val_dataset[index]["audio"]
    processed = preprocess_audio(sample["array"], sample["sampling_rate"])
    assert processed is not None and np.isfinite(processed).all()
    print(f"[{index}] {len(processed)/TARGET_SR:.2f}s | peak={np.max(np.abs(processed)):.3f} | rms={np.sqrt(np.mean(processed**2)):.4f}")

---
## TSK-1.4: Set de calibracion offline

Se reservan 512 clips del split nativo `validation`, balanceados por combinaciones de
acento y genero. Los audios se concatenan para tecnicas que requieran activaciones; los
metadatos conservan offsets, texto y trazabilidad, sin publicar identificadores de hablante.

- `data/calibration_tensor.pt`
- `data/calibration_metadata.json`

In [ ]:
print("[S1-06] Construccion de calibracion estratificada")
N_CALIB_SAMPLES = 512
CALIB_SEED = 123

random.seed(CALIB_SEED)
np.random.seed(CALIB_SEED)

print(f"Seleccionando {N_CALIB_SAMPLES} muestras de calibracion...")

# --- Muestreo estratificado por acento y genero, con candidatos de reserva ---
calib_source = val_dataset
n_source = len(calib_source)
meta_columns = [name for name in ["accent", "gender"] if name in calib_source.column_names]
meta_view = calib_source.select_columns(meta_columns)
calib_groups = defaultdict(list)
for idx in range(n_source):
    row = meta_view[idx]
    key = tuple((row.get(name) or "unknown") for name in meta_columns)
    calib_groups[key].append(idx)
rng_calib = random.Random(CALIB_SEED)
for indices_group in calib_groups.values():
    rng_calib.shuffle(indices_group)
candidate_target = min(n_source, max(N_CALIB_SAMPLES, int(N_CALIB_SAMPLES * 1.25)))
calib_candidate_indices = []
active_groups = list(calib_groups)
rng_calib.shuffle(active_groups)
while active_groups and len(calib_candidate_indices) < candidate_target:
    next_groups = []
    for key in active_groups:
        if calib_groups[key]:
            calib_candidate_indices.append(calib_groups[key].pop())
        if calib_groups[key]:
            next_groups.append(key)
        if len(calib_candidate_indices) >= candidate_target:
            break
    active_groups = next_groups

# --- Preprocesar y recolectar ---
calib_audios = []
calib_metadata = []
rejected = 0
current_offset = 0

for local_idx in tqdm(calib_candidate_indices, desc="Preprocesando calibracion"):
    if len(calib_audios) >= N_CALIB_SAMPLES:
        break
    sample = calib_source[local_idx]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    processed = preprocess_audio(arr, sr)
    if processed is None:
        rejected += 1
        continue

    n_samples = len(processed)
    calib_audios.append(torch.from_numpy(processed))
    calib_metadata.append({
        "local_idx": local_idx,
        "sentence": sample["sentence"],
        "offset_start": current_offset,
        "offset_end": current_offset + n_samples,
        "n_samples": n_samples,
        "duration_s": n_samples / TARGET_SR,
        "gender": sample.get("gender", ""),
        "age": sample.get("age", ""),
        "accent": sample.get("accent", ""),
    })
    current_offset += n_samples

print(f"\nMuestras procesadas : {len(calib_audios)}")
print(f"Muestras rechazadas : {rejected}")
print(f"Total de muestras de audio : {current_offset:,} ({current_offset/TARGET_SR:.1f}s)")
if len(calib_audios) != N_CALIB_SAMPLES:
    raise RuntimeError(f"Calibracion incompleta: {len(calib_audios)}/{N_CALIB_SAMPLES}")

In [ ]:
print("[S1-07] Guardado del tensor de calibracion")
# --- Concatenar en tensor unificado y guardar ---
calibration_tensor = torch.cat(calib_audios, dim=0)

tensor_path = DATA_DIR / "calibration_tensor.pt"
torch.save(calibration_tensor, tensor_path)

metadata_path = DATA_DIR / "calibration_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_id": DATASET_ID,
        "dataset_config": DATASET_CONFIG,
        "dataset_revision": DATASET_REVISION,
        "dataset_fingerprint": val_dataset._fingerprint,
        "split_origin": "validation",
        "target_sr": TARGET_SR,
        "target_db_fs": TARGET_DB_FS,
        "n_samples": len(calib_audios),
        "total_audio_samples": calibration_tensor.shape[0],
        "total_duration_s": calibration_tensor.shape[0] / TARGET_SR,
        "records": calib_metadata,
    }, f, indent=2, ensure_ascii=False)

size_mb = tensor_path.stat().st_size / (1024 * 1024)
print(f"Tensor de calibracion : {tensor_path}")
print(f"Shape                 : {calibration_tensor.shape}")
print(f"Tamaño en disco       : {size_mb:.1f} MB")
print(f"Metadatos             : {metadata_path}")

In [ ]:
print("[S1-08] Verificacion del set de calibracion")
# --- Verificacion de carga ---
print("=== Verificacion del Set de Calibracion ===\n")

loaded_tensor = torch.load(DATA_DIR / "calibration_tensor.pt", weights_only=True)
with open(DATA_DIR / "calibration_metadata.json", "r") as f:
    loaded_meta = json.load(f)

print(f"Tensor cargado  : {loaded_tensor.shape}, dtype={loaded_tensor.dtype}")
print(f"Duracion total  : {loaded_meta['total_duration_s']:.1f}s")
print(f"Numero de clips : {loaded_meta['n_samples']}")
print(f"\nPrimeros 3 clips:")
for rec in loaded_meta["records"][:3]:
    print(f"  idx={rec['local_idx']:>6} | {rec['duration_s']:.2f}s | "
          f"\"{rec['sentence'][:60]}...\"")

# --- Verificar offsets ---
for i, rec in enumerate(loaded_meta["records"]):
    start, end = rec["offset_start"], rec["offset_end"]
    segment = loaded_tensor[start:end]
    assert len(segment) == rec["n_samples"], f"Offset mismatch en indice {i}"

print("\nVerificacion de offsets: OK")

# Sprint 1.5: Primera Adaptacion Monolingue al Espanol

La unica linea canonica de fine-tuning es:

`weights/vibevoice-1.5b` + Common Voice ES -> `outputs/finetune_vibevoice_es/lora` -> `weights/vibevoice-1.5b-es`

## Procedencia del checkpoint existente

El artefacto local `vibevoice-1.5b-es` se obtuvo en la primera ejecucion historica con:

- LoRA rank 8, alpha 32 y dropout 0.05 sobre 196 proyecciones Qwen.
- Fine-tuning completo del diffusion head preentrenado.
- LR `2.5e-5`, CE weight `0.04`, diffusion weight `1.4`, un epoch y 5,263 pasos.
- 18.6 horas en una RTX 4060 Ti; salida FP32 de aproximadamente 10.3 GB.

La implementacion publicada conserva esos hiperparametros para documentar el experimento,
pero corrige integridad de datos, checkpointing, semantica, normalizacion y la mascara CE.
Por ello una ejecucion nueva es una reproduccion metodologica mejorada, no bit a bit.

El intento posterior de aplicar un segundo LoRA CE-only fue rechazado (`WER=1.0`) y no
forma parte de este pipeline. Sus pesos no deben utilizarse.

---
## TSK-1.5.1: Manifiestos train/dev speaker-disjoint

Se extraen hablantes completos desde `train` hasta formar un dev de al menos 2,000 clips.
Esos hablantes se eliminan del entrenamiento. El JSONL se genera localmente porque las
rutas de audio pertenecen al cache de Hugging Face y no son portables entre maquinas.
El split nativo `validation` permanece reservado para calibracion PTQ.

In [ ]:
print("[S1.5-01] Creacion de train/dev speaker-disjoint")
import re, unicodedata

FINETUNE_DATA_DIR = DATA_DIR / "finetune"
FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_JSONL = FINETUNE_DATA_DIR / "cv17_es_train.jsonl"
VAL_JSONL = FINETUNE_DATA_DIR / "cv17_es_val.jsonl"
SPLIT_SEED = 42
VAL_TARGET = min(2000, max(1, len(train_dataset) // 10))

def normalize_spanish_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text or ""))
    text = "".join(char for char in text if char.isprintable())
    text = re.sub(r"\s+", " ", text).strip()
    return text

speaker_indices = defaultdict(list)
for index, client_id in enumerate(train_dataset["client_id"]):
    speaker = client_id or f"missing-speaker-{index}"
    speaker_indices[speaker].append(index)

speakers = list(speaker_indices)
random.Random(SPLIT_SEED).shuffle(speakers)
val_speakers, val_indices_set = [], set()
for speaker in speakers:
    val_speakers.append(speaker)
    val_indices_set.update(speaker_indices[speaker])
    if len(val_indices_set) >= VAL_TARGET:
        break
train_indices = [index for index in range(len(train_dataset)) if index not in val_indices_set]
assert set(train_indices).isdisjoint(val_indices_set)
train_speakers = {train_dataset[index].get("client_id") or f"missing-speaker-{index}" for index in train_indices}
assert train_speakers.isdisjoint(val_speakers)

def write_jsonl(indices, destination: Path) -> tuple[int, int]:
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    written = rejected = 0
    with temporary.open("w", encoding="utf-8") as stream:
        for index in tqdm(indices, desc=f"Escribiendo {destination.name}"):
            sample = train_dataset[index]
            text = normalize_spanish_text(sample.get("sentence"))
            audio_path = sample["audio"].get("path")
            if not text or not audio_path:
                rejected += 1
                continue
            stream.write(json.dumps({"text": f"Speaker 1: {text}", "audio": audio_path}, ensure_ascii=False) + "\n")
            written += 1
    temporary.replace(destination)
    return written, rejected

train_written, train_rejected = write_jsonl(train_indices, TRAIN_JSONL)
val_written, val_rejected = write_jsonl(sorted(val_indices_set), VAL_JSONL)
speaker_hashes = sorted(hashlib.sha256(s.encode()).hexdigest() for s in val_speakers)
split_metadata = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "dataset_fingerprint": train_dataset._fingerprint,
    "seed": SPLIT_SEED,
    "train_records": train_written,
    "validation_records": val_written,
    "validation_speakers": len(val_speakers),
    "validation_speaker_hashes": speaker_hashes,
    "rejected": train_rejected + val_rejected,
}
(FINETUNE_DATA_DIR / "split_metadata.json").write_text(
    json.dumps(split_metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(f"Train: {train_written:,} | Dev: {val_written:,} | Speakers dev: {len(val_speakers):,}")

---
## TSK-1.5.2: Primer LoRA y diffusion head

El script versionado es la fuente autoritativa. Configuracion historica documentada:

| Parametro | Valor |
|---|---:|
| Base | `weights/vibevoice-1.5b` |
| LoRA | r=8, alpha=32, dropout=0.05 |
| Targets | q/k/v/o + gate/up/down |
| Learning rate | `2.5e-5`, cosine, warmup 3% |
| Batch efectivo | 1 x acumulacion 64 |
| CE / diffusion | `0.04` / `1.4` |
| Diffusion head | full fine-tuning |
| Connectors/tokenizers | congelados |
| Epochs | 1 |

La CE actual supervisa continuaciones `speech_diffusion` y su token terminal. La mascara
historica excluia esas continuaciones; esa implementacion no debe restaurarse.

In [ ]:
print("[S1.5-02] Validacion del script de entrenamiento")
import subprocess

train_script_path = SCRIPTS_DIR / "run_finetune_es.sh"
assert train_script_path.is_file(), f"Falta {train_script_path}"
script_text = train_script_path.read_text(encoding="utf-8")
required_fragments = [
    'MODEL_PATH="$PROJECT_ROOT/weights/vibevoice-1.5b"',
    'OUTPUT_DIR="$PROJECT_ROOT/outputs/finetune_vibevoice_es"',
    "--learning_rate 2.5e-5",
    "--diffusion_loss_weight 1.4",
    "--train_diffusion_head True",
    "--ce_loss_weight 0.04",
]
missing = [fragment for fragment in required_fragments if fragment not in script_text]
assert not missing, f"Configuracion historica incompleta en el script: {missing}"
subprocess.run(["bash", "-n", str(train_script_path)], cwd=PROJECT_ROOT, check=True)
print(f"Script valido: {train_script_path}")

In [ ]:
print("[S1.5-03] Ejecucion persistente del primer fine-tuning espanol")
import sys

lora_path = OUTPUTS_DIR / "finetune_vibevoice_es" / "lora"
adapter_config = lora_path / "adapter_config.json"
adapter_weights = [lora_path / "adapter_model.safetensors", lora_path / "adapter_model.bin"]
diffusion_weights = lora_path / "diffusion_head_full.bin"

if adapter_config.is_file() and any(path.is_file() for path in adapter_weights) and diffusion_weights.is_file():
    print(f"[S1.5-03] Artefactos existentes validados: {lora_path}")
else:
    if str(SCRIPTS_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPTS_DIR))
    from persistent_notebook_job import run_persistent_job

    fingerprint_files = [
        SCRIPTS_DIR / "run_finetune_es.sh",
        TRAIN_JSONL,
        VAL_JSONL,
        BASE_MODEL_PATH / "config.json",
        PROJECT_ROOT / "VibeVoice_repo" / "vibevoice" / "finetune" / "train_vibevoice.py",
        PROJECT_ROOT / "VibeVoice_repo" / "vibevoice" / "finetune" / "data_vibevoice.py",
    ]
    fingerprint = hashlib.sha256()
    for source in fingerprint_files:
        fingerprint.update(source.read_bytes())
    run_persistent_job(
        PROJECT_ROOT,
        "s1-5-first-finetune",
        ["bash", str(SCRIPTS_DIR / "run_finetune_es.sh"), sys.executable],
        required_outputs=[adapter_config, diffusion_weights],
        fingerprint=fingerprint.hexdigest(),
    )

assert adapter_config.is_file()
assert any(path.is_file() for path in adapter_weights)
assert diffusion_weights.is_file()
print(f"[S1.5-03] LoRA y diffusion head validados: {lora_path}")

---
## TSK-1.5.3: Merge al modelo base

El primer LoRA del LLM y el diffusion head entrenado se fusionan sobre VibeVoice default.
El staging evita promover checkpoints incompletos. El resultado canonico es
`weights/vibevoice-1.5b-es/`; no existe una segunda rama `-corrected` en este flujo.

In [ ]:
print("[S1.5-04] Validacion autocontenida del script de merge")
import subprocess

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
lora_path = OUTPUTS_DIR / "finetune_vibevoice_es" / "lora"
assert (lora_path / "adapter_config.json").is_file(), f"Falta el LoRA en {lora_path}"
assert (lora_path / "diffusion_head_full.bin").is_file(), f"Falta diffusion head en {lora_path}"
script_text = merge_script_path.read_text(encoding="utf-8")
assert 'BASE_MODEL="$PROJECT_ROOT/weights/vibevoice-1.5b"' in script_text
assert 'MERGE_OUTPUT="$PROJECT_ROOT/weights/vibevoice-1.5b-es"' in script_text
subprocess.run(["bash", "-n", str(merge_script_path)], cwd=PROJECT_ROOT, check=True)
print(f"Script de merge valido: {merge_script_path}")

In [ ]:
print("[S1.5-05] Merge persistente de la primera adaptacion")
import sys

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
model_es_path = WEIGHTS_DIR / "vibevoice-1.5b-es"
complete_model = (model_es_path / "config.json").is_file() and any(model_es_path.glob("*.safetensors"))
if complete_model:
    print(f"[S1.5-05] Checkpoint existente: {model_es_path}")
else:
    if str(SCRIPTS_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPTS_DIR))
    from persistent_notebook_job import run_persistent_job
    merge_fingerprint = hashlib.sha256()
    for source in [
        merge_script_path,
        OUTPUTS_DIR / "finetune_vibevoice_es" / "lora" / "adapter_config.json",
        OUTPUTS_DIR / "finetune_vibevoice_es" / "lora" / "adapter_model.safetensors",
        OUTPUTS_DIR / "finetune_vibevoice_es" / "lora" / "diffusion_head_full.bin",
        BASE_MODEL_PATH / "config.json",
    ]:
        merge_fingerprint.update(source.read_bytes())
    run_persistent_job(
        PROJECT_ROOT,
        "s1-5-first-merge",
        ["bash", str(merge_script_path), sys.executable],
        required_outputs=[model_es_path / "config.json"],
        fingerprint=merge_fingerprint.hexdigest(),
    )

weight_files = sorted(model_es_path.glob("*.safetensors"))
assert (model_es_path / "config.json").is_file() and weight_files
print(f"[S1.5-05] VibeVoice-ES validado: {len(weight_files)} shard(s) en {model_es_path}")

---
## TSK-1.5.4: Carga y gate linguistico

Primero se verifica que `VibeVoice-ES` cargue en BF16 dentro de 8 GB. Luego se genera el
panel espanol en el modo oficial de clonacion/prefill, usando una voz versionada del demo.
Whisper large-v3 calcula WER/CER y los WAV quedan disponibles para revision auditiva.

In [ ]:
print("[S1.5-06] Carga de VibeVoice-ES")
import gc, sys

repo_path = PROJECT_ROOT / "VibeVoice_repo"
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration

MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es"
assert (MODEL_ES_PATH / "config.json").is_file() and any(MODEL_ES_PATH.glob("*.safetensors")), (
    "Falta VibeVoice-ES. Complete S1.5-03 y S1.5-05."
)
if "model_es" in globals():
    del model_es
gc.collect()
torch.cuda.empty_cache()
model_es = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
    local_files_only=True,
)
vram_es = torch.cuda.memory_allocated() / 1024**3
model_size_mib = sum(path.stat().st_size for path in MODEL_ES_PATH.glob("*.safetensors")) / 1024**2
print(f"VibeVoice-ES cargado | VRAM={vram_es:.2f} GiB | disco={model_size_mib:.1f} MiB")

In [ ]:
print("[S1.5-07] Gate linguistico de VibeVoice-ES")
import gc, json, sys

MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es"
quality_dir = OUTPUTS_DIR / "sprint1_5_quality_gate"
voice_sample = PROJECT_ROOT / "VibeVoice_repo" / "demo" / "voices" / "en-Alice_woman.wav"
quality_script = SCRIPTS_DIR / "validate_vibevoice_es_quality.py"
assert voice_sample.is_file(), f"Falta la voz versionada: {voice_sample}"

if "model_es" in globals():
    del model_es
gc.collect()
torch.cuda.empty_cache()

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
from quality_gate import validate_quality_gate

fingerprint = hashlib.sha256()
for source in [MODEL_ES_PATH / "config.json", *sorted(MODEL_ES_PATH.glob("*.safetensors")), voice_sample, quality_script]:
    with source.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            fingerprint.update(chunk)
command = [
    sys.executable,
    str(quality_script),
    "--model", str(MODEL_ES_PATH),
    "--output", str(quality_dir),
    "--max-wer", "0.30",
    "--voice-sample", str(voice_sample),
    "--seed", "42",
    "--asr-device", "cpu",
]
metrics_path = quality_dir / "metrics.json"
try:
    quality_metrics = validate_quality_gate(MODEL_ES_PATH, metrics_path)
    print("[S1.5-07] Gate existente ligado al checkpoint actual")
except (FileNotFoundError, ValueError):
    run_persistent_job(
        PROJECT_ROOT,
        "s1-5-quality-gate-first-lora",
        command,
        required_outputs=[metrics_path],
        fingerprint=fingerprint.hexdigest(),
    )
    quality_metrics = validate_quality_gate(MODEL_ES_PATH, metrics_path)
print(f"WER={quality_metrics['wer']:.4f} | CER={quality_metrics['cer']:.4f} | RTF={quality_metrics['rtf']:.3f}")
print(f"Revise auditivamente: {quality_dir}")

---
# Sprint 2: TorchAO W4A16 HQQ g128 selectivo

> `torchao==0.18.0` con kernel INT4 nativo de PyTorch/CUDA.
> Solo las 196 proyecciones de Qwen se cuantizan a INT4 por grupos de 128.
> Activaciones, audio, connectors, embeddings, `lm_head` y prediction head permanecen BF16.
> El preflight instala `torchao==0.18.0` con `--no-deps` si falta; no cuantiza ni crea pesos.
> El build no se ejecuta automaticamente: requiere ejecutar separadamente la celda S2-02.
> Salida candidata: `weights/vibevoice-1.5b-es-torchao-int4/`.

Este sprint reemplaza GGUF/CrispASR porque su ruta C++ de voz condicionada no reproduce el checkpoint canonico. TorchAO conserva el procesador y la inferencia oficial de VibeVoice, y solo promueve el artefacto si la recarga limpia y el gate `WER <= 0.30` pasan con `en-Alice_woman.wav`.

In [2]:
print("[S2-01] Instalacion y preflight TorchAO W4A16 selectivo")
import importlib
import importlib.metadata
import json
import subprocess
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es"
TORCHAO_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-torchao-int4"
TORCHAO_VERSION = "0.18.0"
try:
    installed_torchao = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    installed_torchao = None
if installed_torchao != TORCHAO_VERSION:
    print(f"Instalando torchao=={TORCHAO_VERSION} sin alterar PyTorch ni sus dependencias...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "--no-deps", f"torchao=={TORCHAO_VERSION}",
    ], check=True)
    importlib.invalidate_caches()
    installed_torchao = importlib.metadata.version("torchao")
if installed_torchao != TORCHAO_VERSION:
    raise RuntimeError(f"Version TorchAO invalida: {installed_torchao}; requerida: {TORCHAO_VERSION}")

import torch
from torchao.quantization import Int4WeightOnlyConfig
from torchao.quantization.quantize_.workflows import Int4ChooseQParamsAlgorithm, Int4PackingFormat

if not torch.cuda.is_available(): raise RuntimeError("Sprint 2 requiere una GPU CUDA visible desde este kernel")
if torch.cuda.get_device_capability(0)[0] < 8: raise RuntimeError("TorchAO INT4 requiere compute capability >= 8.0")
ao_config = Int4WeightOnlyConfig(
    group_size=128,
    int4_packing_format=Int4PackingFormat.TILE_PACKED_TO_4D,
    int4_choose_qparams_algorithm=Int4ChooseQParamsAlgorithm.HQQ,
)
assert ao_config.group_size == 128
assert ao_config.int4_packing_format.value == "tile_packed_to_4d"
assert ao_config.int4_choose_qparams_algorithm.value == "hqq"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
from quantization_common import sha256
cfg = json.loads((MODEL_ES_PATH / "config.json").read_text(encoding="utf-8"))
decoder = cfg["decoder_config"]
assert decoder["hidden_size"] == 1536 and decoder["num_hidden_layers"] == 28
assert decoder["num_attention_heads"] == 12 and decoder["num_key_value_heads"] == 2
print(f"Preflight aprobado: TorchAO {installed_torchao}, {torch.cuda.get_device_name(0)}, Qwen2 28 capas, 196 matrices objetivo")
print("Esta celda instala la dependencia si falta, pero no cuantiza ni crea pesos.")

[S2-01] Instalacion y preflight TorchAO W4A16 selectivo
Instalando torchao==0.18.0 sin alterar PyTorch ni sus dependencias...
  Using cached torchao-0.18.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (21 kB)
Using cached torchao-0.18.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (3.4 MB)


W0808 19:56:59.548000 21169 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Preflight aprobado: TorchAO 0.18.0, NVIDIA GeForce RTX 4060 Ti, Qwen2 28 capas, 196 matrices objetivo
Esta celda instala la dependencia si falta, pero no cuantiza ni crea pesos.


In [1]:
print("[S2-02] Build, recarga y gate TorchAO W4A16")
print("ATENCION: esta celda cuantiza el checkpoint. Ejecutar solo con autorizacion explicita.")
import importlib.metadata
import json
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es"
TORCHAO_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-torchao-int4"
TORCHAO_PARTIAL = WEIGHTS_DIR / "vibevoice-1.5b-es-torchao-int4.partial"
TORCHAO_METRICS = OUTPUTS_DIR / "sprint2_torchao_int4" / "metrics.json"
try:
    torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError as exc:
    raise RuntimeError("Ejecute primero S2-01 para instalar y validar TorchAO") from exc
if torchao_version != "0.18.0": raise RuntimeError(f"TorchAO invalido: {torchao_version}; ejecute S2-01")
if TORCHAO_PARTIAL.exists(): print(f"Se reanudara el candidato guardado sin volver a cuantizar: {TORCHAO_PARTIAL}")
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
from quality_gate import validate_quality_gate
from quantization_common import sha256
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
fingerprint = {
    "source_hashes": quality_gate["source_hashes"],
    "implementation": sha256(SCRIPTS_DIR / "torchao_quantize_vibevoice.py"),
    "torchao_version": torchao_version,
}
run_persistent_job(
    PROJECT_ROOT, "s2-torchao-int4",
    [sys.executable, str(SCRIPTS_DIR / "torchao_quantize_vibevoice.py")],
    required_outputs=[TORCHAO_OUTPUT / "manifest.json", TORCHAO_METRICS],
    fingerprint=json.dumps(fingerprint, sort_keys=True),
)
print(f"Artefacto aprobado: {TORCHAO_OUTPUT}")

[S2-02] Build, recarga y gate TorchAO W4A16
ATENCION: esta celda cuantiza el checkpoint. Ejecutar solo con autorizacion explicita.
Se reanudara el candidato guardado sin volver a cuantizar: /home/alfrog/projects/VibeVoice_Optimization/weights/vibevoice-1.5b-es-torchao-int4.partial
[s2-torchao-int4] log anterior archivado en /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s2-torchao-int4/job-1786238105.log
[s2-torchao-int4] iniciado como PID 64289. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s2-torchao-int4/job.log
[s2-torchao-int4] completado. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s2-torchao-int4/job.log
Artefacto aprobado: /home/alfrog/projects/VibeVoice_Optimization/weights/vibevoice-1.5b-es-torchao-int4


In [2]:
print("[S2-03] Verificacion del resultado TorchAO")
import json
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from quantization_common import sha256, source_hashes
artifact = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-torchao-int4"
metrics_path = PROJECT_ROOT / "outputs" / "sprint2_torchao_int4" / "metrics.json"
manifest_path = artifact / "manifest.json"
if not manifest_path.is_file(): raise FileNotFoundError(f"No existe un artefacto TorchAO aprobado: {manifest_path}")
if not metrics_path.is_file(): raise FileNotFoundError(f"Faltan las metricas de Sprint 2: {metrics_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
metrics = manifest["validation"]["quality"]
saved_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
artifact_files = [p for p in artifact.rglob("*") if p.is_file() and p.name != "manifest.json"]
artifact_inventory = {str(p.relative_to(artifact)): {"bytes": p.stat().st_size, "sha256": sha256(p)} for p in sorted(artifact_files)}
assert manifest["schema"] == "vibevoice-selective-torchao-int4-v1"
assert manifest["status"] == "validated"
assert manifest["source_hashes"] == source_hashes(PROJECT_ROOT / "weights" / "vibevoice-1.5b-es")
assert manifest["quantization"]["type"] == "W4A16"
assert manifest["quantization"]["algorithm"] == "hqq"
assert manifest["quantization"]["group_size"] == 128
assert manifest["quantization"]["packing_format"] == "tile_packed_to_4d"
assert manifest["quantization"]["torchao_version"] == "0.18.0"
assert manifest["artifacts"] == artifact_inventory
assert manifest["validation"]["reload_verified"] is True
assert manifest["validation"]["quantized_modules"] == 196
assert metrics["passed"] and metrics["wer"] <= metrics["max_wer"]
assert saved_metrics == metrics
print(f"TorchAO aprobado: WER={metrics['wer']:.4f}, RTF={metrics['rtf']:.4f}, pico={metrics['vram_peak_gib']:.2f} GiB")

[S2-03] Verificacion del resultado TorchAO
TorchAO aprobado: WER=0.1845, RTF=1.0978, pico=3.55 GiB


---
# Sprint 3: GPTQ W4 g128 selectivo y recargable

> GPTQModel 2.2.0 + Transformers 4.51.3 + Triton 3.7.0 en el entorno `vibevoice`.
> Solo las 196 proyecciones Qwen usan GPTQ W4; audio, connectors, embeddings, `lm_head` y prediction head quedan BF16.
> El build usa staging reanudable y nunca elimina automaticamente candidatos o artefactos finales.
> La aceptacion exige recarga limpia y el gate canonico de seis frases, voz fija y `WER <= 0.30`.

In [1]:
print("[S3-01] Preflight GPTQModel/Triton selectivo")
import importlib.metadata
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR = PROJECT_ROOT / "data"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es"
GPTQ_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-gptq-tts"
GPTQ_PARTIAL = WEIGHTS_DIR / "vibevoice-1.5b-es-gptq-tts.partial"
GPTQ_METADATA = DATA_DIR / "calibration_metadata.json"
required_versions = {"gptqmodel": "2.2.0", "transformers": "4.51.3", "triton": "3.7.0"}
installed_versions = {}
for package, expected in required_versions.items():
    try:
        installed_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError("Falta el stack GPTQ; ejecute bash scripts/setup_gptqmodel.sh") from exc
    if installed_versions[package] != expected:
        raise RuntimeError(f"Version invalida de {package}: {installed_versions[package]}; requerida: {expected}")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from gptq_vibevoice import preflight as validate_gptq_environment
from quality_gate import validate_quality_gate
environment = validate_gptq_environment()
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
calibration_info = json.loads(GPTQ_METADATA.read_text(encoding="utf-8"))
records = calibration_info.get("records", [])
texts = [record.get("sentence", "").strip() for record in records]
assert calibration_info.get("split_origin") == "validation"
assert calibration_info.get("n_samples") == 512 and len(records) == 512
assert all(texts) and len(set(texts)) == 512
model_config = json.loads((MODEL_ES_PATH / "config.json").read_text(encoding="utf-8"))
assert model_config["decoder_config"]["tie_word_embeddings"] is True
free_vram, total_vram = torch.cuda.mem_get_info()
if free_vram < 6 * 1024**3: raise RuntimeError(f"VRAM libre insuficiente: {free_vram / 1024**3:.2f} GiB")
if shutil.disk_usage(WEIGHTS_DIR).free < 20 * 1024**3: raise RuntimeError("Espacio en disco insuficiente para staging GPTQ")
whisper_cache = Path.home() / ".cache" / "whisper" / "large-v3.pt"
if not whisper_cache.is_file(): raise FileNotFoundError(f"Falta Whisper large-v3 cacheado: {whisper_cache}")
if GPTQ_PARTIAL.exists(): print(f"Se reanudara el staging GPTQ existente: {GPTQ_PARTIAL}")
print(f"Preflight aprobado: {environment['gpu']}, 512 registros, 196 objetivos, {free_vram / 1024**3:.2f} GiB libres")
print("Esta celda solo ejecuta un kernel Triton sintetico; no cuantiza VibeVoice.")

[S3-01] Preflight GPTQModel/Triton selectivo
No broken requirements found.


/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0809 10:22:43.041000 2420 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.



INFO  ENV: Auto setting PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


INFO  Optimize: `TritonV2QuantLinear` compilation triggered.                   
{
  "torch": "2.12.0+cu130",
  "transformers": "4.51.3",
  "gptqmodel": "2.2.0",
  "triton": "3.7.0",
  "gpu": "NVIDIA GeForce RTX 4060 Ti",
  "compute_capability": [
    8,
    9
  ],
  "triton_kernel": "passed"
}
Preflight aprobado: NVIDIA GeForce RTX 4060 Ti, 512 registros, 196 objetivos, 6.84 GiB libres
Esta celda solo ejecuta un kernel Triton sintetico; no cuantiza VibeVoice.


In [1]:
print("[S3-02] Extraccion BF16, calibracion y cuantizacion GPTQ")
print("ATENCION: esta celda cuantiza el checkpoint. Ejecutar solo con autorizacion explicita.")
import importlib.metadata
import json
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es"
GPTQ_OUTPUT = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-gptq-tts"
GPTQ_PARTIAL = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-gptq-tts.partial"
GPTQ_METADATA = PROJECT_ROOT / "data" / "calibration_metadata.json"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from gptq_vibevoice import validate
from persistent_notebook_job import run_persistent_job
from quality_gate import validate_quality_gate
from quantization_common import sha256
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
if GPTQ_PARTIAL.exists(): print(f"Se reanudara el staging existente sin eliminarlo: {GPTQ_PARTIAL}")
fingerprint = {
    "source_hashes": quality_gate["source_hashes"],
    "metadata": sha256(GPTQ_METADATA),
    "implementation": sha256(SCRIPTS_DIR / "gptq_vibevoice.py"),
    "launcher": sha256(SCRIPTS_DIR / "build_vibevoice_gptq_tts.sh"),
    "gptqmodel": importlib.metadata.version("gptqmodel"),
    "transformers": importlib.metadata.version("transformers"),
    "triton": importlib.metadata.version("triton"),
    "config": "W4-g128-sym-desc_act_true-tts_prefill_256",
}
run_persistent_job(
    PROJECT_ROOT, "s3-gptq-build", [
        "bash", str(SCRIPTS_DIR / "build_vibevoice_gptq_tts.sh"),
        str(MODEL_ES_PATH), str(GPTQ_OUTPUT), str(GPTQ_METADATA),
    ],
    required_outputs=[GPTQ_OUTPUT / "manifest.json"],
    fingerprint=json.dumps(fingerprint, sort_keys=True),
)
gptq_manifest = validate(GPTQ_OUTPUT)
assert gptq_manifest["status"] in {"structure_validated", "validated"}
assert gptq_manifest["validation"]["gptq_modules"] == 196
print(f"Modelo GPTQ guardado y validado estructuralmente: {GPTQ_OUTPUT}")

[S3-02] Extraccion BF16, calibracion y cuantizacion GPTQ
ATENCION: esta celda cuantiza el checkpoint. Ejecutar solo con autorizacion explicita.


/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0809 10:31:04.052000 6692 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.



INFO  ENV: Auto setting PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


Se reanudara el staging existente sin eliminarlo: /home/alfrog/projects/VibeVoice_Optimization/weights/vibevoice-1.5b-es-gptq.partial
[s3-gptq-build] log anterior archivado en /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s3-gptq-build/job-1786289474.log
[s3-gptq-build] iniciado como PID 6769. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s3-gptq-build/job.log
[s3-gptq-build] PID 6769 activo. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s3-gptq-build/job.log
[s3-gptq-build] completado. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s3-gptq-build/job.log
{
  "gptq_modules": 196,
  "protected_tensors": 866,
  "total_bytes": 3502464892
}
Modelo GPTQ guardado y validado estructuralmente: /home/alfrog/projects/VibeVoice_Optimization/weights/vibevoice-1.5b-es-gptq


In [ ]:
print("[S3-03] Recarga limpia y gate TTS del artefacto GPTQ")
import json
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
GPTQ_OUTPUT = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-gptq-tts"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from gptq_vibevoice import validate
from persistent_notebook_job import run_persistent_job
from quantization_common import sha256
gptq_manifest = validate(GPTQ_OUTPUT)
GPTQ_SMOKE_WAV = OUTPUTS_DIR / "sprint3_gptq_tts" / "smoke-es.wav"
fingerprint = {
    "artifacts": gptq_manifest["artifacts"],
    "implementation": sha256(SCRIPTS_DIR / "smoke_vibevoice_gptq.py"),
    "protocol": "six-texts-en-Alice-large-v3-WER-0.30-v1",
}
run_persistent_job(
    PROJECT_ROOT, "s3-gptq-smoke", [
        sys.executable, str(SCRIPTS_DIR / "smoke_vibevoice_gptq.py"),
        "--model", str(GPTQ_OUTPUT), "--output", str(GPTQ_SMOKE_WAV),
        "--max-wer", "0.30",
    ],
    required_outputs=[GPTQ_SMOKE_WAV, GPTQ_SMOKE_WAV.with_suffix(".json")],
    fingerprint=json.dumps(fingerprint, sort_keys=True),
)
gptq_smoke = json.loads(GPTQ_SMOKE_WAV.with_suffix(".json").read_text(encoding="utf-8"))
assert gptq_smoke["gptq_modules"] == 196
assert len(gptq_smoke["samples"]) == 6
assert gptq_smoke["voice"].endswith("en-Alice_woman.wav")
validated_manifest = json.loads((GPTQ_OUTPUT / "manifest.json").read_text(encoding="utf-8"))
if gptq_smoke["passed"]:
    assert gptq_smoke["wer"] <= 0.30 and validated_manifest["status"] == "validated"
    print(f"GPTQ aprobado: WER={gptq_smoke['wer']:.4f}, RTF={gptq_smoke['rtf']:.4f}, pico={gptq_smoke['vram_peak_gib']:.2f} GiB")
else:
    assert gptq_smoke["wer"] > 0.30 and validated_manifest["status"] == "quality_failed"
    print(f"GPTQ rechazado por calidad: WER={gptq_smoke['wer']:.4f} > 0.3000")

## Sprint 3R: revision GPTQ con activaciones TTS-prefill reales

> Conserva GPTQ v1 y construye un candidato independiente con 256 prefills de voz, 32589 tokens efectivos y `desc_act=True`.
> El build medido tomo 415.47 s (0.115 h). El gate mejoro WER de 0.5516 a 0.3423, pero el candidato permanece rechazado porque supera 0.30.
> La cuantizacion solo debe iniciarse con autorizacion explicita; ningun artefacto se elimina automaticamente.

In [ ]:
print("[S3R-02/03] Build, recarga limpia y gate GPTQ TTS-prefill")
print("ATENCION: el build cuantiza el checkpoint si el candidato aun no existe.")
import json
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es"
GPTQ_TTS_OUTPUT = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-gptq-tts"
GPTQ_TTS_SMOKE = OUTPUTS_DIR / "sprint3_gptq_tts" / "smoke-es.wav"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from gptq_vibevoice import validate
from persistent_notebook_job import run_persistent_job
from quality_gate import validate_quality_gate
from quantization_common import sha256
validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
build_fingerprint = json.dumps({
    "implementation": sha256(SCRIPTS_DIR / "gptq_vibevoice.py"),
    "launcher": sha256(SCRIPTS_DIR / "build_vibevoice_gptq_tts.sh"),
    "config": "W4-g128-sym-desc_act_true-tts_prefill_256",
}, sort_keys=True)
run_persistent_job(
    PROJECT_ROOT, "s3r-gptq-tts-build",
    ["bash", str(SCRIPTS_DIR / "build_vibevoice_gptq_tts.sh")],
    required_outputs=[GPTQ_TTS_OUTPUT / "manifest.json"],
    fingerprint=build_fingerprint,
)
manifest = validate(GPTQ_TTS_OUTPUT)
assert manifest["validation"]["gptq_modules"] == 196
smoke_fingerprint = json.dumps({
    "artifacts": manifest["artifacts"],
    "implementation": sha256(SCRIPTS_DIR / "smoke_vibevoice_gptq.py"),
    "protocol": "six-texts-en-Alice-large-v3-WER-0.30-v1",
}, sort_keys=True)
run_persistent_job(
    PROJECT_ROOT, "s3r-gptq-tts-smoke", [
        sys.executable, str(SCRIPTS_DIR / "smoke_vibevoice_gptq.py"),
        "--model", str(GPTQ_TTS_OUTPUT), "--output", str(GPTQ_TTS_SMOKE),
        "--max-wer", "0.30",
    ],
    required_outputs=[GPTQ_TTS_SMOKE, GPTQ_TTS_SMOKE.with_suffix(".json")],
    fingerprint=smoke_fingerprint,
)
metrics = json.loads(GPTQ_TTS_SMOKE.with_suffix(".json").read_text(encoding="utf-8"))
final_manifest = json.loads((GPTQ_TTS_OUTPUT / "manifest.json").read_text(encoding="utf-8"))
assert metrics["gptq_modules"] == 196 and len(metrics["samples"]) == 6
assert metrics["passed"] == (metrics["wer"] <= 0.30)
assert final_manifest["status"] == ("validated" if metrics["passed"] else "quality_failed")
print(f"GPTQ TTS-prefill: WER={metrics['wer']:.4f}, CER={metrics['cer']:.4f}, RTF={metrics['rtf']:.4f}, pico={metrics['vram_peak_gib']:.2f} GiB")

---
# Sprint 4: AWQ W4A16 selectivo y recargable

> AutoAWQ 0.2.9: W4A16 asimetrico, zero_point=True, grupo 128, GEMM/Triton.
> Solo las 196 proyecciones Qwen se cuantizan; audio, connectors y diffusion quedan FP16.
> El artefacto se acepta tras validar pesos, metadatos, ejecucion de las 196 capas y WER.
> No se etiqueta como Marlin: el artefacto ncoder-ai real tambien es GEMM y este host no tiene awq_ext.


In [3]:
print("[S4-01] Preflight AWQ selectivo")
import importlib.metadata
import json
import sys
from pathlib import Path
import torch

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR = PROJECT_ROOT / "data"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es"
AWQ_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-awq"
AWQ_METADATA = DATA_DIR / "calibration_metadata.json"
assert importlib.metadata.version("autoawq") == "0.2.9", "Ejecute bash scripts/setup_autoawq.sh"
assert importlib.metadata.version("transformers") == "4.51.3"
assert (MODEL_ES_PATH / "config.json").is_file(), (
    "Falta el checkpoint canonico aprobado por S1.5-07"
)
with AWQ_METADATA.open(encoding="utf-8") as file:
    calibration_info = json.load(file)
records = calibration_info.get("records", [])
texts = [record.get("sentence", "").strip() for record in records]
assert calibration_info.get("split_origin") == "validation"
assert calibration_info.get("n_samples") == 512 and len(records) == 512
assert all(texts) and len(set(texts)) == 512
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from awq_vibevoice import preflight as validate_awq_environment
from quality_gate import validate_quality_gate
environment = validate_awq_environment()
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
free_vram, _ = torch.cuda.mem_get_info()
if free_vram < 6 * 1024**3: raise RuntimeError(f"VRAM libre insuficiente: {free_vram / 1024**3:.2f} GiB")
print(f"AutoAWQ GEMM/Triton aprobado en {environment['gpu']}; 256 prefills TTS, 196 proyecciones Qwen")


[S4-01] Preflight AWQ selectivo


/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.

{
  "torch": "2.12.0+cu130",
  "transformers": "4.51.3",
  "autoawq": "0.2.9",
  "triton": "3.7.0",
  "gpu": "NVIDIA GeForce RTX 4060 Ti",
  "compute_capability": [
    8,
    9
  ],
  "triton_kernel": "passed"
}
AutoAWQ GEMM/Triton aprobado en NVIDIA GeForce RTX 4060 Ti; 256 prefills TTS, 196 proyecciones Qwen


In [4]:
print("[S4-02] Extraccion FP16, calibracion y cuantizacion AWQ")
import importlib.metadata
import json, sys
from pathlib import Path
notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR, OUTPUTS_DIR = PROJECT_ROOT / "scripts", PROJECT_ROOT / "outputs"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es"
AWQ_OUTPUT = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-awq"
AWQ_METADATA = PROJECT_ROOT / "data" / "calibration_metadata.json"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
from persistent_notebook_job import run_persistent_job
from quantization_common import sha256
from quantization_common import sha256
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
fingerprint = {
    "source_hashes": quality_gate["source_hashes"],
    "metadata": sha256(AWQ_METADATA),
    "implementation": sha256(SCRIPTS_DIR / "awq_vibevoice.py"),
    "launcher": sha256(SCRIPTS_DIR / "build_vibevoice_awq.sh"),
    "autoawq": importlib.metadata.version("autoawq"),
    "triton": importlib.metadata.version("triton"),
    "config": "W4A16-g128-asym-gemm-tts_prefill_256",
}
run_persistent_job(
    PROJECT_ROOT, "s4-awq-build", [
        "bash", str(SCRIPTS_DIR / "build_vibevoice_awq.sh"),
        str(MODEL_ES_PATH), str(AWQ_OUTPUT), str(AWQ_METADATA),
    ],
    required_outputs=[AWQ_OUTPUT / "manifest.json"],
    fingerprint=json.dumps(fingerprint, sort_keys=True),
)
with (AWQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    awq_manifest = json.load(file)
assert awq_manifest["status"] in {"structure_validated", "validated", "quality_failed"}
assert awq_manifest["validation"]["awq_modules"] == 196
print(f"Modelo AWQ guardado: {AWQ_OUTPUT}")


[S4-02] Extraccion FP16, calibracion y cuantizacion AWQ
[s4-awq-build] iniciado como PID 51945. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s4-awq-build/job.log


RuntimeError: [s4-awq-build] fallo con codigo 1. Log:
/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/audioread/rawread.py:16: DeprecationWarning: 'aifc' is deprecated and slated for removal in Python 3.13
  import aifc
/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/audioread/rawread.py:17: DeprecationWarning: 'audioop' is deprecated and slated for removal in Python 3.13
  import audioop
/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/audioread/rawread.py:19: DeprecationWarning: 'sunau' is deprecated and slated for removal in Python 3.13
  import sunau
Prepared FP16 decoder and protected VibeVoice modules in /home/alfrog/projects/VibeVoice_Optimization/weights/vibevoice-1.5b-es-awq.partial
W0809 12:05:21.090000 52157 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is de

In [ ]:
print("[S4-03] Recarga limpia y gate canonico TTS del artefacto AWQ")
import json, sys
from pathlib import Path
notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR, OUTPUTS_DIR = PROJECT_ROOT / "scripts", PROJECT_ROOT / "outputs"
AWQ_OUTPUT = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-awq"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
from quantization_common import sha256
awq_manifest = json.loads((AWQ_OUTPUT / "manifest.json").read_text(encoding="utf-8"))
AWQ_SMOKE_WAV = OUTPUTS_DIR / "sprint4_awq" / "smoke-es.wav"
run_persistent_job(
    PROJECT_ROOT, "s4-awq-smoke", [
        sys.executable, str(SCRIPTS_DIR / "smoke_vibevoice_awq.py"),
        "--model", str(AWQ_OUTPUT), "--output", str(AWQ_SMOKE_WAV),
    ],
    required_outputs=[AWQ_SMOKE_WAV, AWQ_SMOKE_WAV.with_suffix(".json")],
    fingerprint=json.dumps({
        "artifacts": awq_manifest["artifacts"],
        "implementation": sha256(SCRIPTS_DIR / "smoke_vibevoice_awq.py"),
        "protocol": "six-texts-en-Alice-large-v3-WER-0.30-v1",
    }, sort_keys=True),
)
assert AWQ_SMOKE_WAV.is_file() and AWQ_SMOKE_WAV.stat().st_size > 44
with AWQ_SMOKE_WAV.with_suffix(".json").open(encoding="utf-8") as file:
    awq_smoke = json.load(file)
assert awq_smoke["awq_modules"] == 196
with (AWQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    final_manifest = json.load(file)
assert len(awq_smoke["samples"]) == 6 and awq_smoke["max_wer"] == 0.30
assert final_manifest["status"] == ("validated" if awq_smoke["passed"] else "quality_failed")
print(json.dumps(awq_smoke, indent=2))


---
# Sprint 5: INT8 Hibrido Selectivo via bitsandbytes

> Replica el enfoque de Fabio Sarracino sobre VibeVoice 1.5B: cuantiza unicamente
> las capas `nn.Linear` del decoder Qwen2 con `BitsAndBytesConfig(load_in_8bit=True)`.
> El cabezal de difusion (`prediction_head` en este fork), los tokenizers, los
> connectors y `lm_head` permanecen en BF16 para proteger los latentes acusticos.

No se usa split layout: convertir el `state_dict` INT8 a FP16 destruiria la
cuantizacion. `save_pretrained()` conserva directamente los pesos y metadatos bnb.


In [5]:
print("[S5-INT8] Ejecucion persistente fuera del kernel")
import json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts" / "sprint5_int8.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
from persistent_notebook_job import run_persistent_job
from quantization_common import sha256
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
metrics_path = OUTPUTS_DIR / "sprint5_int8" / "metrics.json"
run_persistent_job(
    PROJECT_ROOT,
    "s5-int8",
    [sys.executable, str(SCRIPTS_DIR / "sprint5_int8.py")],
    required_outputs=[metrics_path, PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-int8" / "manifest.json"],
    workdir=PROJECT_ROOT / "notebooks",
    fingerprint=json.dumps(quality_gate["source_hashes"], sort_keys=True),
)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))


[S5-INT8] Ejecucion persistente fuera del kernel
[s5-int8] iniciado como PID 61813. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s5-int8/job.log
[s5-int8] completado. Log: /home/alfrog/projects/VibeVoice_Optimization/outputs/.notebook-jobs/s5-int8/job.log
{
  "model": "/home/alfrog/projects/VibeVoice_Optimization/weights/vibevoice-1.5b-es-int8",
  "source": "/home/alfrog/projects/VibeVoice_Optimization/weights/vibevoice-1.5b-es",
  "source_hashes": {
    "config.json": "b0d6db52f45dd5179b42217c55112aa2ee66e7ad7b40b0950833c9f6780a0f21",
    "model-00001-of-00003.safetensors": "4844b0b79cca5ea029af6757429997a73fc26f55be5119213b46c58a37d69a2f",
    "model-00002-of-00003.safetensors": "4cb2604093b04077dda058ccb0a3e8179fe00b3de31abd9fd43a3206d7fd5a1b",
    "model-00003-of-00003.safetensors": "28e299f427bc930886b16517ecbdf180bc54a341e79af7698dc2a5c1bebeff11"
  },
  "source_gate_wer": 0.23214285714285712,
  "quantization": "int8",
  "quantized_modules": 196,
  "qua

---
# Sprint 6: NF4 Hibrido + Double Quant via bitsandbytes

> Cuantiza unicamente las 196 proyecciones lineales del Qwen2 con NF4 y doble
> cuantizacion. `embed_tokens`, `lm_head` y todo el pipeline acustico permanecen
> en BF16; las multiplicaciones NF4 tambien usan BF16 como compute dtype.

`embed_tokens` es `nn.Embedding`, por lo que bitsandbytes no lo reemplazaria de
todos modos; se incluye en la lista para documentar y verificar la proteccion.
No se usa split layout ni se exporta un state dict dequantizado a FP16.


In [ ]:
print("[S6-NF4] Ejecucion persistente fuera del kernel")
import json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts" / "sprint6_nf4.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
from persistent_notebook_job import run_persistent_job
from quantization_common import sha256
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
metrics_path = OUTPUTS_DIR / "sprint6_nf4" / "metrics.json"
run_persistent_job(
    PROJECT_ROOT,
    "s6-nf4",
    [sys.executable, str(SCRIPTS_DIR / "sprint6_nf4.py")],
    required_outputs=[metrics_path, PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-nf4.partial" / "manifest.json"],
    workdir=PROJECT_ROOT / "notebooks",
    fingerprint=json.dumps({
        "source_hashes": quality_gate["source_hashes"],
        "implementation": sha256(SCRIPTS_DIR / "bnb_quantize_vibevoice.py"),
        "launcher": sha256(SCRIPTS_DIR / "sprint6_nf4.py"),
        "protocol": "six-texts-en-Alice-large-v3-WER-0.30-v1",
    }, sort_keys=True),
)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
if not metrics["passed"]: print("NF4 no fue promovido porque no aprobo WER <= 0.30")
print(json.dumps(metrics, indent=2))


---
# Sprint 7: SmoothQuant W8A8 selectivo sobre Qwen2

> Las 196 proyecciones lineales de Qwen almacenan pesos INT8 con escalas SmoothQuant.
> Cada forward cuantiza dinamicamente las activaciones y ejecuta `torch._int_mm`.
> La salida es BF16. Audio, embeddings y `lm_head` permanecen en BF16.

El loader custom esta en `scripts/smoothquant_vibevoice.py`; alpha=0.5 y solo
se cuantifica el decoder Qwen.


In [ ]:
print("[S7-SmoothQuant] Ejecucion persistente fuera del kernel")
import json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts" / "sprint7_smoothquant.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SMOOTHQUANT_MODEL = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-smoothquant"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
from persistent_notebook_job import run_persistent_job
from quantization_common import sha256
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
metrics_path = OUTPUTS_DIR / "sprint7_smoothquant" / "metrics.json"
run_persistent_job(
    PROJECT_ROOT,
    "s7-smoothquant",
    [sys.executable, str(SCRIPTS_DIR / "sprint7_smoothquant.py")],
    required_outputs=[metrics_path],
    workdir=PROJECT_ROOT / ".",
    fingerprint=json.dumps({
        "source_hashes": quality_gate["source_hashes"],
        "implementation": sha256(SCRIPTS_DIR / "sprint7_smoothquant.py"),
        "backend": sha256(SCRIPTS_DIR / "smoothquant_vibevoice.py"),
        "protocol": "six-texts-en-Alice-large-v3-WER-0.30-v1",
    }, sort_keys=True),
)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
assert metrics["quantized_modules"] == 196
if metrics["passed"]:
    assert (SMOOTHQUANT_MODEL / "manifest.json").is_file()
else:
    assert (SMOOTHQUANT_MODEL.with_name(SMOOTHQUANT_MODEL.name + ".partial") / "manifest.json").is_file()
print(json.dumps(metrics, indent=2))


---
# Sprint 8: Benchmark Final (7 Modelos + Estadistica)

> 200 muestras del split test de CV17.
> Metricas: VRAM, RTF, WER, CER, PESQ, MCD, Disco.
> Shapiro-Wilk + Wilcoxon + Graficos + Tabla LaTeX.


In [ ]:
print("[S8] Benchmark final: siete variantes en procesos limpios")
import json, subprocess, sys
from pathlib import Path
notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts" / "sprint8_benchmark.py").is_file()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "sprint8_benchmark.py")], cwd=PROJECT_ROOT, check=True)
summary = json.loads((PROJECT_ROOT / "outputs" / "sprint8_benchmark" / "summary.json").read_text(encoding="utf-8"))
assert summary["models_total"] == 7 and summary["models_completed"] == 7
print(json.dumps(summary, indent=2, ensure_ascii=False))
